In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os

In [ ]:
df = pd.read_csv(os.path.join('data', 'health_costs.csv'))
df.head()

In [ ]:
df = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=False, dtype=float)
df.head()

In [ ]:
train_dataset = df.sample(frac=0.8, random_state=0)
test_dataset = df.drop(train_dataset.index)

train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

In [ ]:
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(np.array(train_dataset))

model = tf.keras.Sequential([
    normalizer,
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mae',
    metrics=['mae', 'mse']
)

model.summary()

In [ ]:
history = model.fit(
    train_dataset,
    train_labels,
    epochs=200,
    validation_split=0.2,
    verbose=1
)

In [ ]:
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)
print(f"Test MAE: {mae:.2f}")